# 11 Temporal Diagnosability Analysis

This notebook tests whether the 10 milestone's 120 s improvement is already using protection-action information. It compares all available samples with a strict pre-protection-only scope at 10, 20, 30, 40, 60, 90, and 120 s. Strict eligibility requires a parsed first protection time and an actual last sampled time strictly before that event; unknown protection times are excluded.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.multi_accident_features import FIRST_VERSION_CLASSES, input_feature_groups
from src.temporal_diagnosability import (
    SCOPES,
    TEMPORAL_WINDOWS_S,
    build_temporal_dataset,
    evaluate_temporal_models,
    selected_test_row,
    select_validation_model,
)

RESULT_ROOT = PROJECT_ROOT / 'results'
FIGURE_ROOT = RESULT_ROOT / 'figures'
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
CLASS_LABELS = list(FIRST_VERSION_CLASSES)
TARGET_CLASSES = ['FLB', 'LLB', 'LOCAC', 'RW', 'RI', 'SGBTR', 'LOCA']


## Build the temporal dataset and enforce protection-time eligibility

In [2]:
feature_groups, removed_features = input_feature_groups()
assert len(feature_groups['A_strict_38']) == 38
dataset, window_points = build_temporal_dataset(
    project_root=PROJECT_ROOT, windows_s=TEMPORAL_WINDOWS_S, classes=tuple(CLASS_LABELS)
)
strict = dataset.loc[dataset['strict_pre_protection']].copy()
assert strict['first_protection_s'].notna().all()
assert (strict['first_protection_s'] > strict['last_sample_s']).all()
assert dataset[['sample_id', 'window_s']].duplicated().sum() == 0
assert set(dataset['window_s']) == set(TEMPORAL_WINDOWS_S)

all_counts = (
    dataset.groupby(['window_s', 'accident_class'], as_index=False)['sample_id']
    .nunique().assign(scope='all_sample')
)
strict_counts = (
    strict.groupby(['window_s', 'accident_class'], as_index=False)['sample_id']
    .nunique().assign(scope='strict_pre_protection_only')
)
class_counts = pd.concat([all_counts, strict_counts], ignore_index=True).rename(columns={'sample_id': 'trajectory_count'})
window_counts = (
    class_counts.groupby(['scope', 'window_s'], as_index=False)['trajectory_count']
    .sum()
)
unknown_counts = (
    dataset.loc[dataset['first_protection_s'].isna()]
    .groupby('window_s')['sample_id'].nunique()
)

window_points.to_csv(RESULT_ROOT / '11_temporal_window_points.csv', index=False)
class_counts.to_csv(RESULT_ROOT / '11_temporal_class_counts.csv', index=False)
window_counts.to_csv(RESULT_ROOT / '11_temporal_window_eligibility.csv', index=False)
print('All-sample counts:', window_counts.query("scope == 'all_sample'").set_index('window_s')['trajectory_count'].to_dict())
print('Strict counts:', window_counts.query("scope == 'strict_pre_protection_only'").set_index('window_s')['trajectory_count'].to_dict())
print('Unknown first-protection trajectories by window:', unknown_counts.to_dict())
display(class_counts.query("accident_class in @TARGET_CLASSES").sort_values(['scope', 'window_s', 'accident_class']))


All-sample counts: {10: 1211, 20: 1211, 30: 1211, 40: 1211, 60: 1211, 90: 1211, 120: 1193}
Strict counts: {10: 660, 20: 660, 30: 660, 40: 660, 60: 659, 90: 659, 120: 471}
Unknown first-protection trajectories by window: {10: 551, 20: 551, 30: 551, 40: 551, 60: 551, 90: 551, 120: 533}


,window_s,accident_class,trajectory_count,scope
0,10,FLB,100,all_sample
1,10,LLB,101,all_sample
2,10,LOCA,100,all_sample
3,10,LOCAC,100,all_sample
6,10,RI,100,all_sample
...,...,...,...,...
133,120,LOCA,76,strict_pre_protection_only
134,120,LOCAC,79,strict_pre_protection_only
135,120,RI,28,strict_pre_protection_only
136,120,RW,3,strict_pre_protection_only


## All-sample versus strict pre-protection metrics

In [3]:
metrics, per_class, confusion = evaluate_temporal_models(
    dataset, feature_groups, CLASS_LABELS, windows_s=TEMPORAL_WINDOWS_S
)
assert set(metrics['scope']) == set(SCOPES)
assert set(metrics['window_s']) == set(TEMPORAL_WINDOWS_S)
assert metrics[['accuracy', 'macro_f1', 'macro_f1_observed_classes', 'balanced_accuracy']].notna().all().all()
strict_metrics = metrics.loc[metrics['scope'].eq('strict_pre_protection_only')]
assert (strict_metrics['n_unknown_protection'] == 0).all()
assert len(confusion) == len(metrics) * len(CLASS_LABELS) * len(CLASS_LABELS)
assert confusion.groupby(['scope', 'window_s', 'input_group', 'model', 'eval_split'])['count'].sum().ge(1).all()

metrics.to_csv(RESULT_ROOT / '11_temporal_metrics.csv', index=False)
per_class.to_csv(RESULT_ROOT / '11_temporal_per_class.csv', index=False)
confusion.to_csv(RESULT_ROOT / '11_temporal_confusion_matrices.csv', index=False)
display(metrics.query("input_group == 'A_strict_38' and eval_split == 'test'").sort_values(['scope', 'window_s', 'macro_f1'], ascending=[True, True, False]))


,scope,window_s,input_group,model,eval_split,n_samples,n_features,n_classes_train,n_classes_eval,n_unknown_protection,accuracy,macro_f1,macro_f1_observed_classes,balanced_accuracy,fit_status
3,all_sample,10,A_strict_38,logistic_regression,test,123,190,12,12,59,0.276423,0.192755,0.192755,0.272727,fitted
5,all_sample,10,A_strict_38,random_forest,test,123,190,12,12,59,0.276423,0.192755,0.192755,0.272727,fitted
7,all_sample,10,A_strict_38,hist_gradient_boosting,test,123,190,12,12,59,0.276423,0.192755,0.192755,0.272727,fitted
1,all_sample,10,A_strict_38,naive_majority,test,123,190,12,12,59,0.081301,0.012531,0.012531,0.083333,majority_baseline
27,all_sample,20,A_strict_38,logistic_regression,test,123,190,12,12,59,0.276423,0.192755,0.192755,0.272727,fitted
29,all_sample,20,A_strict_38,random_forest,test,123,190,12,12,59,0.276423,0.192755,0.192755,0.272727,fitted
31,all_sample,20,A_strict_38,hist_gradient_boosting,test,123,190,12,12,59,0.276423,0.192755,0.192755,0.272727,fitted
25,all_sample,20,A_strict_38,naive_majority,test,123,190,12,12,59,0.081301,0.012531,0.012531,0.083333,majority_baseline
51,all_sample,30,A_strict_38,logistic_regression,test,123,190,12,12,59,0.276423,0.192755,0.192755,0.272727,fitted
53,all_sample,30,A_strict_38,random_forest,test,123,190,12,12,59,0.276423,0.192755,0.192755,0.272727,fitted


## Leakage and SLBIC initial-condition sensitivity

In [4]:
test_metrics = metrics.loc[metrics['eval_split'].eq('test')].copy()
base = test_metrics.loc[test_metrics['input_group'].eq('A_strict_38')].set_index(['scope', 'window_s', 'model'])
sensitivity_rows = []
for row in test_metrics.itertuples(index=False):
    reference = base.loc[(row.scope, row.window_s, row.model)]
    slbic_recall = per_class.loc[
        per_class['scope'].eq(row.scope) & per_class['window_s'].eq(row.window_s)
        & per_class['input_group'].eq(row.input_group) & per_class['model'].eq(row.model)
        & per_class['eval_split'].eq('test') & per_class['accident_class'].eq('SLBIC'), 'recall'
    ].iloc[0]
    base_slbic = per_class.loc[
        per_class['scope'].eq(row.scope) & per_class['window_s'].eq(row.window_s)
        & per_class['input_group'].eq('A_strict_38') & per_class['model'].eq(row.model)
        & per_class['eval_split'].eq('test') & per_class['accident_class'].eq('SLBIC'), 'recall'
    ].iloc[0]
    sensitivity_rows.append({
        'scope': row.scope, 'window_s': row.window_s, 'model': row.model, 'input_group': row.input_group,
        'test_macro_f1': row.macro_f1, 'test_macro_f1_observed_classes': row.macro_f1_observed_classes,
        'test_balanced_accuracy': row.balanced_accuracy, 'slbic_recall': slbic_recall,
        'delta_macro_f1_vs_A': row.macro_f1 - reference.macro_f1,
        'delta_balanced_accuracy_vs_A': row.balanced_accuracy - reference.balanced_accuracy,
        'delta_slbic_recall_vs_A': slbic_recall - base_slbic,
    })
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(RESULT_ROOT / '11_temporal_sensitivity_comparison.csv', index=False)
display(sensitivity.query("model == 'hist_gradient_boosting'").sort_values(['scope', 'window_s', 'input_group']))


,scope,window_s,model,input_group,test_macro_f1,test_macro_f1_observed_classes,test_balanced_accuracy,slbic_recall,delta_macro_f1_vs_A,delta_balanced_accuracy_vs_A,delta_slbic_recall_vs_A
3,all_sample,10,hist_gradient_boosting,A_strict_38,0.192755,0.192755,0.272727,0.272727,0.000000,0.000000,0.000000
7,all_sample,10,hist_gradient_boosting,B_without_potential_leakage,0.192755,0.192755,0.272727,0.272727,0.000000,0.000000,0.000000
11,all_sample,10,hist_gradient_boosting,C_without_SLBIC_initial,0.192755,0.192755,0.272727,0.272727,0.000000,0.000000,0.000000
15,all_sample,20,hist_gradient_boosting,A_strict_38,0.192755,0.192755,0.272727,0.272727,0.000000,0.000000,0.000000
19,all_sample,20,hist_gradient_boosting,B_without_potential_leakage,0.192755,0.192755,0.272727,0.272727,0.000000,0.000000,0.000000
23,all_sample,20,hist_gradient_boosting,C_without_SLBIC_initial,0.192755,0.192755,0.272727,0.272727,0.000000,0.000000,0.000000
27,all_sample,30,hist_gradient_boosting,A_strict_38,0.192755,0.192755,0.272727,0.272727,0.000000,0.000000,0.000000
31,all_sample,30,hist_gradient_boosting,B_without_potential_leakage,0.192755,0.192755,0.272727,0.272727,0.000000,0.000000,0.000000
35,all_sample,30,hist_gradient_boosting,C_without_SLBIC_initial,0.192755,0.192755,0.272727,0.272727,0.000000,0.000000,0.000000
39,all_sample,40,hist_gradient_boosting,A_strict_38,0.192755,0.192755,0.272727,0.272727,0.000000,0.000000,0.000000


## Selected recall traces and figures

In [5]:
selected_metric_rows = []
selected_recall_rows = []
for scope in SCOPES:
    for input_group in feature_groups:
        for window_s in TEMPORAL_WINDOWS_S:
            selection = select_validation_model(metrics, scope, input_group, window_s)
            test_row = selected_test_row(metrics, selection)
            selected_metric_rows.append(test_row.to_dict())
            for accident_class in TARGET_CLASSES:
                recall_row = per_class.loc[
                    per_class['scope'].eq(scope) & per_class['window_s'].eq(window_s)
                    & per_class['input_group'].eq(input_group) & per_class['model'].eq(selection['model'])
                    & per_class['eval_split'].eq('test') & per_class['accident_class'].eq(accident_class)
                ].iloc[0]
                selected_recall_rows.append({
                    'scope': scope, 'input_group': input_group, 'window_s': window_s,
                    'selected_model': selection['model'], 'accident_class': accident_class,
                    'recall': recall_row['recall'], 'support': recall_row['support'],
                    'test_macro_f1': test_row['macro_f1'],
                    'test_balanced_accuracy': test_row['balanced_accuracy'],
                })
selected_metrics = pd.DataFrame(selected_metric_rows)
selected_recall = pd.DataFrame(selected_recall_rows)
selected_metrics.to_csv(RESULT_ROOT / '11_temporal_selected_metrics.csv', index=False)
selected_recall.to_csv(RESULT_ROOT / '11_temporal_selected_target_recall.csv', index=False)

all_120_ids = set(dataset.loc[dataset['window_s'].eq(120), 'sample_id'])
strict_120_ids = set(dataset.loc[dataset['window_s'].eq(120) & dataset['strict_pre_protection'], 'sample_id'])
all_120_cohort = dataset.loc[dataset['sample_id'].isin(all_120_ids) & dataset['window_s'].isin([60, 120])].copy()
strict_120_cohort = dataset.loc[dataset['sample_id'].isin(strict_120_ids) & dataset['window_s'].isin([60, 120])].copy()
matched_all_metrics, _, _ = evaluate_temporal_models(
    all_120_cohort, {'A_strict_38': feature_groups['A_strict_38']}, CLASS_LABELS, windows_s=(60, 120)
)
matched_strict_metrics, _, _ = evaluate_temporal_models(
    strict_120_cohort, {'A_strict_38': feature_groups['A_strict_38']}, CLASS_LABELS, windows_s=(60, 120)
)
matched_metrics = pd.concat([
    matched_all_metrics.loc[matched_all_metrics['scope'].eq('all_sample')].assign(matched_cohort='all_120_available'),
    matched_strict_metrics.loc[matched_strict_metrics['scope'].eq('strict_pre_protection_only')].assign(matched_cohort='strict_120_pre_protection'),
], ignore_index=True)
matched_metrics.to_csv(RESULT_ROOT / '11_temporal_matched_cohort_metrics.csv', index=False)
print('Matched all-120 cohort:', len(all_120_ids), 'trajectories')
print('Matched strict-120 pre-protection cohort:', len(strict_120_ids), 'trajectories')

fig, ax = plt.subplots(figsize=(10, 5))
for scope, frame in window_counts.groupby('scope'):
    frame = frame.sort_values('window_s')
    ax.plot(frame['window_s'], frame['trajectory_count'], marker='o', label=scope)
ax.set_xlabel('Window (s)')
ax.set_ylabel('Trajectory count')
ax.set_title('All-sample versus strict pre-protection eligibility')
ax.set_xticks(TEMPORAL_WINDOWS_S)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '11_temporal_scope_counts.png', dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for scope, ax, title in zip(SCOPES, axes, ['All-sample', 'Strict pre-protection only']):
    frame = selected_metrics.query("input_group == 'A_strict_38' and scope == @scope").sort_values('window_s')
    ax.plot(frame['window_s'], frame['macro_f1'], marker='o', label='Macro-F1')
    ax.plot(frame['window_s'], frame['balanced_accuracy'], marker='s', label='Balanced Accuracy')
    ax.set_title(title)
    ax.set_xlabel('Window (s)')
    ax.set_ylabel('Test score')
    ax.set_xticks(TEMPORAL_WINDOWS_S)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()
fig.suptitle('Temporal diagnosability: selected group-A model per window')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '11_temporal_all_vs_strict_performance.png', dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for scope, ax, title in zip(SCOPES, axes, ['All-sample', 'Strict pre-protection only']):
    frame = selected_recall.query("input_group == 'A_strict_38' and scope == @scope")
    for accident_class, class_frame in frame.groupby('accident_class'):
        class_frame = class_frame.sort_values('window_s')
        ax.plot(class_frame['window_s'], class_frame['recall'], marker='o', label=accident_class)
    ax.set_title(title)
    ax.set_xlabel('Window (s)')
    ax.set_xticks(TEMPORAL_WINDOWS_S)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.25)
axes[0].set_ylabel('Test per-class Recall')
axes[0].legend(fontsize=8, ncol=2)
fig.suptitle('Target-class Recall over time')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '11_temporal_target_recall.png', dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for scope, ax, title in zip(SCOPES, axes, ['All-sample', 'Strict pre-protection only']):
    frame = sensitivity.query("scope == @scope and model == 'hist_gradient_boosting'")
    for input_group in ['B_without_potential_leakage', 'C_without_SLBIC_initial']:
        group_frame = frame.loc[frame['input_group'].eq(input_group)].sort_values('window_s')
        ax.plot(group_frame['window_s'], group_frame['delta_macro_f1_vs_A'], marker='o', label=input_group)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel('Window (s)')
    ax.set_ylabel('Macro-F1 delta versus A')
    ax.set_xticks(TEMPORAL_WINDOWS_S)
    ax.grid(alpha=0.25)
axes[0].legend(fontsize=8)
fig.suptitle('Leakage and SLBIC initial-condition sensitivity')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / '11_temporal_sensitivity.png', dpi=150)
plt.close(fig)


Matched all-120 cohort: 1193 trajectories
Matched strict-120 pre-protection cohort: 471 trajectories


In [6]:
for scope in SCOPES:
    for window_s in TEMPORAL_WINDOWS_S:
        selection = select_validation_model(metrics, scope, 'A_strict_38', window_s)
        table = confusion.loc[
            confusion['scope'].eq(scope) & confusion['window_s'].eq(window_s)
            & confusion['input_group'].eq('A_strict_38') & confusion['model'].eq(selection['model'])
            & confusion['eval_split'].eq('test')
        ]
        matrix = table.pivot(index='true_class', columns='predicted_class', values='count').reindex(
            index=CLASS_LABELS, columns=CLASS_LABELS, fill_value=0
        ).to_numpy()
        fig, ax = plt.subplots(figsize=(9, 7))
        image = ax.imshow(matrix, cmap='Blues', vmin=0)
        for i in range(len(CLASS_LABELS)):
            for j in range(len(CLASS_LABELS)):
                ax.text(j, i, int(matrix[i, j]), ha='center', va='center', fontsize=7)
        ax.set_xticks(range(len(CLASS_LABELS)), CLASS_LABELS, rotation=60, ha='right')
        ax.set_yticks(range(len(CLASS_LABELS)), CLASS_LABELS)
        ax.set_xlabel('Predicted class')
        ax.set_ylabel('True class')
        ax.set_title(f"{scope}, {window_s}s, {selection['model']}")
        fig.colorbar(image, ax=ax, label='Count')
        fig.tight_layout()
        filename_scope = 'all_sample' if scope == 'all_sample' else 'strict_pre_protection_only'
        fig.savefig(FIGURE_ROOT / f'11_confusion_{filename_scope}_{window_s}s.png', dpi=150)
        plt.close(fig)


## Summary and interpretation gates

In [7]:
late_gain_rows = []
for model in sorted(metrics['model'].unique()):
    row = {'model': model}
    for scope in SCOPES:
        for window_s in [60, 120]:
            value = metrics.loc[
                metrics['scope'].eq(scope) & metrics['input_group'].eq('A_strict_38')
                & metrics['model'].eq(model) & metrics['eval_split'].eq('test')
                & metrics['window_s'].eq(window_s), 'macro_f1'
            ].iloc[0]
            row[f'{scope}_{window_s}_macro_f1'] = value
        row[f'{scope}_macro_f1_gain_120_vs_60'] = row[f'{scope}_120_macro_f1'] - row[f'{scope}_60_macro_f1']
    late_gain_rows.append(row)
late_gain = pd.DataFrame(late_gain_rows)
late_gain.to_csv(RESULT_ROOT / '11_temporal_120_vs_60_gain.csv', index=False)

matched_gain_rows = []
for model in sorted(matched_metrics['model'].unique()):
    row = {'model': model}
    for cohort in ['all_120_available', 'strict_120_pre_protection']:
        frame = matched_metrics.loc[(matched_metrics['matched_cohort'].eq(cohort)) & matched_metrics['model'].eq(model) & matched_metrics['eval_split'].eq('test')]
        row[f'{cohort}_60_macro_f1'] = frame.loc[frame['window_s'].eq(60), 'macro_f1'].iloc[0]
        row[f'{cohort}_120_macro_f1'] = frame.loc[frame['window_s'].eq(120), 'macro_f1'].iloc[0]
        row[f'{cohort}_macro_f1_gain_120_vs_60'] = row[f'{cohort}_120_macro_f1'] - row[f'{cohort}_60_macro_f1']
    matched_gain_rows.append(row)
matched_gain = pd.DataFrame(matched_gain_rows)
matched_gain.to_csv(RESULT_ROOT / '11_temporal_matched_120_vs_60_gain.csv', index=False)

hgb_gain = late_gain.loc[late_gain['model'].eq('hist_gradient_boosting')].iloc[0]
hgb_matched_gain = matched_gain.loc[matched_gain['model'].eq('hist_gradient_boosting')].iloc[0]
all_gain = float(hgb_gain['all_sample_macro_f1_gain_120_vs_60'])
strict_gain = float(hgb_gain['strict_pre_protection_only_macro_f1_gain_120_vs_60'])
matched_all_gain = float(hgb_matched_gain['all_120_available_macro_f1_gain_120_vs_60'])
matched_strict_gain = float(hgb_matched_gain['strict_120_pre_protection_macro_f1_gain_120_vs_60'])
if matched_strict_gain > 0.10:
    late_gain_conclusion = 'The matched strict pre-protection cohort also improves strongly at 120 s; the gain cannot be attributed mainly to post-protection information. The larger all-sample gain is confounded by class/support composition and still requires external validation.'
elif matched_all_gain > 0.10 and matched_strict_gain <= 0.05:
    late_gain_conclusion = 'The matched all-sample cohort improves at 120 s while the matched strict pre-protection cohort does not; post-protection information is the leading explanation.'
else:
    late_gain_conclusion = 'The matched all-sample and strict pre-protection comparisons do not cleanly separate the source of the 120 s gain; class coverage and external validation remain necessary.'

def json_safe(value):
    if value is None:
        return None
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    return value

summary = {
    'experiment': 'temporal diagnosability analysis',
    'classes': CLASS_LABELS,
    'target_classes': TARGET_CLASSES,
    'windows_s': list(TEMPORAL_WINDOWS_S),
    'scope_definition': {
        'all_sample': 'Every trajectory-window with enough sampled data, regardless of parsed first protection time.',
        'strict_pre_protection_only': 'Only trajectories with known first_protection_s and first_protection_s > actual last sampled time in the window.',
        'unknown_first_protection_excluded_from_strict': True,
    },
    'feature_groups': {key: {'feature_count': len(value), 'features': value, 'removed_features': removed_features.get(key, [])} for key, value in feature_groups.items()},
    'window_counts': window_counts.to_dict(orient='records'),
    'unknown_protection_counts': [{'window_s': int(key), 'trajectory_count': int(value)} for key, value in unknown_counts.items()],
    'selected_group_A_test_metrics': selected_metrics.query("input_group == 'A_strict_38'").to_dict(orient='records'),
    'selected_target_recall': selected_recall.query("input_group == 'A_strict_38'").to_dict(orient='records'),
    'late_gain_120_vs_60': late_gain.to_dict(orient='records'),
    'matched_cohort_counts': {'all_120_available': len(all_120_ids), 'strict_120_pre_protection': len(strict_120_ids)},
    'matched_late_gain_120_vs_60': matched_gain.to_dict(orient='records'),
    'late_gain_conclusion': late_gain_conclusion,
    'risk_notes': [
        'Strict pre-protection metrics have reduced and uneven class support because unknown first protection times are excluded.',
        'The same trajectory/sample_id split is used across windows and scopes; time rows are never randomly split.',
        'A/B/C leakage and SLBIC initial-condition sensitivity is reported separately for every window and scope.',
        'A strong 120 s all-sample score is not a valid early-diagnosis claim unless strict pre-protection and external validation agree.',
    ],
}
with open(RESULT_ROOT / '11_temporal_diagnosability_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(json_safe(summary), handle, ensure_ascii=False, indent=2)

assert (RESULT_ROOT / '11_temporal_metrics.csv').exists()
assert (RESULT_ROOT / '11_temporal_per_class.csv').exists()
assert (RESULT_ROOT / '11_temporal_confusion_matrices.csv').exists()
assert (RESULT_ROOT / '11_temporal_diagnosability_summary.json').exists()
assert (RESULT_ROOT / '11_temporal_matched_cohort_metrics.csv').exists()
assert (RESULT_ROOT / '11_temporal_matched_120_vs_60_gain.csv').exists()
assert (FIGURE_ROOT / '11_temporal_all_vs_strict_performance.png').exists()
assert (FIGURE_ROOT / '11_temporal_target_recall.png').exists()
assert (FIGURE_ROOT / '11_temporal_sensitivity.png').exists()
print('HGB all-sample 120-vs-60 Macro-F1 gain:', all_gain)
print('HGB strict pre-protection 120-vs-60 Macro-F1 gain:', strict_gain)
print('HGB matched all-120 cohort 120-vs-60 Macro-F1 gain:', matched_all_gain)
print('HGB matched strict-120 cohort 120-vs-60 Macro-F1 gain:', matched_strict_gain)
print('Conclusion:', late_gain_conclusion)
print('FULL TEMPORAL DIAGNOSABILITY ANALYSIS PASSED')


HGB all-sample 120-vs-60 Macro-F1 gain: 0.5598290064377475
HGB strict pre-protection 120-vs-60 Macro-F1 gain: 0.37143658810325475
HGB matched all-120 cohort 120-vs-60 Macro-F1 gain: 0.5558607524694935
HGB matched strict-120 cohort 120-vs-60 Macro-F1 gain: 0.36494792890141725
Conclusion: The matched strict pre-protection cohort also improves strongly at 120 s; the gain cannot be attributed mainly to post-protection information. The larger all-sample gain is confounded by class/support composition and still requires external validation.
FULL TEMPORAL DIAGNOSABILITY ANALYSIS PASSED
